In [1]:
!pip install yfinance


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

In [3]:
df = yf.download("AAPL", period="6mo", interval="1d")

if isinstance(df.columns, pd.MultiIndex):
    df.columns = [col[0] for col in df.columns]

df = df.reset_index()
df = df.sort_values("Date")

[*********************100%***********************]  1 of 1 completed


In [4]:
df["sma_10"] = df["Close"].rolling(10).mean()
df["sma_20"] = df["Close"].rolling(20).mean()

delta = df["Close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()
rs = avg_gain / avg_loss
df["rsi_14"] = 100 - (100 / (1 + rs))

ema12 = df["Close"].ewm(span=12, adjust=False).mean()
ema26 = df["Close"].ewm(span=26, adjust=False).mean()
df["macd"] = ema12 - ema26
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()

rolling_mean = df["Close"].rolling(20).mean()
rolling_std  = df["Close"].rolling(20).std()
df["bb_upper"] = rolling_mean + (2 * rolling_std)
df["bb_lower"] = rolling_mean - (2 * rolling_std)
df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["Close"]

df["return"]     = df["Close"].pct_change()
df["return_3d"]  = df["Close"].pct_change(3)
df["return_5d"]  = df["Close"].pct_change(5)

df["volatility_5d"]  = df["return"].rolling(5).std()
df["volatility_10d"] = df["return"].rolling(10).std()

df["dist_sma_10"] = (df["Close"] - df["sma_10"]) / df["sma_10"]
df["dist_sma_20"] = (df["Close"] - df["sma_20"]) / df["sma_20"]

df["target"] = (df["return"].shift(-1) > 0.002).astype(int)
df = df.dropna()

In [5]:
feature_cols = [
    "rsi_14", "macd", "macd_signal", "bb_width",
    "return_3d", "return_5d", "volatility_5d", "volatility_10d",
    "dist_sma_10", "dist_sma_20", "Volume"
]

X = df[feature_cols]
y = df["target"]

split_index = int(len(df) * 0.8)

X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

In [6]:
hgb = HistGradientBoostingClassifier(
    max_iter=500,
    max_depth=4,
    learning_rate=0.05,
    min_samples_leaf=10,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=42
)
hgb.fit(X_train, y_train)

HistGradientBoostingClassifier(early_stopping=True, learning_rate=0.05,
                               max_depth=4, max_iter=500, min_samples_leaf=10,
                               n_iter_no_change=20, random_state=42)

In [7]:
pred = hgb.predict(X_test)

print("HistGBM Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, hgb.predict_proba(X_test)[:, 1]))

HistGBM Accuracy: 0.4090909090909091
              precision    recall  f1-score   support

           0       1.00      0.07      0.13        14
           1       0.38      1.00      0.55         8

    accuracy                           0.41        22
   macro avg       0.69      0.54      0.34        22
weighted avg       0.77      0.41      0.29        22

ROC-AUC: 0.65625


In [9]:
prob = hgb.predict_proba(X_test.iloc[[-1]])[:, 1][0]

if prob > 0.6:
    signal = "BUY"
elif prob < 0.4:
    signal = "SELL"
else:
    signal = "HOLD"

print(f"Signal:     {signal}")
print(f"Confidence: {round(prob, 2)}")

Signal:     HOLD
Confidence: 0.54
